# Ch 17 — 작은 벤치마크 만들기

HellaSwag-tiny, 도메인 probe, pass@k, LLM-as-judge 로 본 책 10M 모델의 능력을 측정합니다.

In [ ]:
# 필요 시 설치
# !pip install torch tokenizers anthropic

import torch
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'device: {device}')

## 1. 개념 — "벤치마크" 가 측정하는 것

PPL 은 **언어 모델의 평균 손실**. 벤치마크는 **특정 능력**.

| 벤치마크 | 측정 | 형식 |
|---|---|---|
| **HellaSwag** | 상식 추론 (다음 일 예측) | 4지선다 |
| **MMLU** | 일반 지식 | 4지선다 |
| **HumanEval** | 코드 생성 | 함수 작성 + 테스트 |
| **(본인 도메인)** | 사용 케이스 | 자유 |

## 3. 어디에 쓰이나 — 3가지 도구

### 도구 1. Likelihood-based 4지선다 (HellaSwag 스타일)

In [ ]:
# hellaswag_lite.py
@torch.no_grad()
def score_choice(model, tok, context, choice):
    """context+choice 의 choice 부분 평균 logp."""
    full = context + choice
    ids = torch.tensor([tok.encode(full).ids], device=device)
    ctx_len = len(tok.encode(context).ids)
    logits, _ = model(ids[:, :-1])                                     # shift 1 -- language modeling 표준
    logp = F.log_softmax(logits, dim=-1)
    target = ids[:, 1:]
    # choice 부분만
    choice_logp = logp[0, ctx_len-1:].gather(1, target[0, ctx_len-1:].unsqueeze(1)).mean()
    return choice_logp.item()

def predict(model, tok, item):
    scores = [score_choice(model, tok, item['context'], c) for c in item['choices']]
    return int(torch.tensor(scores).argmax())                          # logp 가장 큰 (PPL 가장 낮은) 선택지

print("score_choice / predict 함수 정의 완료")
print("사용: predict(model, tok, item) -> 0~3 중 예측 인덱스")

In [ ]:
# story_probe.py -- 도메인 probe
PROBES = [
    {
        "prompt": "Once upon a time, there was a little girl named",
        "expect": ["Lily", "Mia", "Sara", "Anna"],         # 인명 자연스러움
        "type": "name_continuation"
    },
    {
        "prompt": "The dog was very happy because",
        "expect_keywords": ["food", "play", "friend", "walk"],   # 합리적 이유
        "type": "causal_completion"
    },
    {
        "prompt": "...and they all lived",
        "expect_exact": "happily ever after",                    # 정형 표현
        "type": "formulaic"
    },
    # 30개 ...
]

def check(output, probe):
    """probe 통과 여부 간단 체크."""
    if "expect_exact" in probe:
        return probe["expect_exact"].lower() in output.lower()
    if "expect" in probe:
        return any(name in output for name in probe["expect"])
    if "expect_keywords" in probe:
        return any(kw in output.lower() for kw in probe["expect_keywords"])
    return False

def evaluate_probes(model, tok, probes, n=5):
    results = {"correct": 0, "total": len(probes), "by_type": {}}
    for p in probes:
        passes = 0
        for _ in range(n):                                             # pass@n
            out = generate(model, tok, p["prompt"], max_tokens=20)
            if check(out, p): passes += 1
        if passes > 0: results["correct"] += 1
        results["by_type"].setdefault(p["type"], [0, 0])
        results["by_type"][p["type"]][1] += 1
        if passes > 0: results["by_type"][p["type"]][0] += 1
    return results

print(f"PROBES 정의 완료: {len(PROBES)}개 (30개로 확장 예정)")

In [ ]:
# pass_at_k.py
def pass_at_k(model, tok, problems, k=5):
    correct = 0
    for prob in problems:
        passes = 0
        for _ in range(k):
            out = generate(model, tok, prob["prompt"], temperature=0.8)
            if check(out, prob["test"]): passes += 1
        if passes > 0: correct += 1
    return correct / len(problems)

print("pass_at_k 함수 정의 완료")
print("사용: pass_at_k(model, tok, problems, k=5) -> 0.0~1.0")

## 4. 최소 예제 — HellaSwag-tiny 30 문항

In [ ]:
# hellaswag_tiny_stories.py
HELLASWAG_TINY = [
    {
        "context": "Lily picked up the apple. She wanted to eat it. She",
        "choices": [
            "took a big bite.",                          # 정답
            "threw it at the moon.",
            "wrote a letter to her dad.",
            "started dancing in the rain.",
        ],
        "answer": 0,
    },
    {
        "context": "The dog saw a cat. The cat was scared. The dog",
        "choices": [
            "wagged his tail and said hello.",          # 정답
            "ate a sandwich.",
            "drove a car to the park.",
            "studied for the math test.",
        ],
        "answer": 0,
    },
    # 28개 더 추가 ...
]

def run_hellaswag_tiny(model, tok):
    correct = 0
    for item in HELLASWAG_TINY:
        pred = predict(model, tok, item)
        if pred == item["answer"]: correct += 1
    return correct / len(HELLASWAG_TINY)

# 모델이 있을 때:
# acc = run_hellaswag_tiny(model, tok)
# print(f"HellaSwag-tiny accuracy: {acc:.1%}")

print(f"HellaSwag-tiny 문항 수: {len(HELLASWAG_TINY)}개 (30개로 확장 권장)")
print()
print("본 책 10M 동화 모델 예상 결과:")
print("  HellaSwag-tiny accuracy: 65.0%   (30 중 19~22 정답)")
print()
print("해석:")
print("  무작위 추측: 25%")
print("  65%: 학습 도메인 (TinyStories) 에서 상식 추론 가능")
print("  진짜 HellaSwag (10K+, 일반 텍스트) 은 같은 모델이 30% 미만 -- 도메인 외")

## 5. 실전 — LLM-as-judge

In [ ]:
# llm_judge.py
# !pip install anthropic  # 필요 시

JUDGE_PROMPT = """동화 모델의 다음 출력을 0~5 점으로 평가:

PROMPT: {prompt}
OUTPUT: {output}

평가 기준:
- 문법 자연스러움
- 인물·사건 일관성
- 어린이용 어휘 적합

점수만 출력 (정수). 출력:"""

def judge(prompt, output):
    import anthropic
    client = anthropic.Anthropic()
    msg = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=8,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(prompt=prompt, output=output)}]
    )
    try:
        return int(msg.content[0].text.strip())
    except:
        return 0

# 50 샘플 평가
# scores = []
# for p in prompts:
#     out = generate(model, tok, p)
#     scores.append(judge(p, out))
# print(f"평균: {sum(scores)/len(scores):.2f}")

print("judge 함수 정의 완료 (ANTHROPIC_API_KEY 필요)")
print()
print("LLM judge 함정:")
print("  1. Self-bias -- 같은 회사 모델이 평가하면 후함")
print("  2. Position bias -- A/B 비교에서 첫 번째에 후함. 항상 랜덤 swap")
print("  3. Length bias -- 긴 답에 후함")
print("  4. Cost -- Haiku 한 번 약 $0.0001. 100 샘플 = $0.01")
print("  5. Drift -- 같은 모델 버전이라도 다른 시점에 다른 답. 재현용 판본 고정")

## 연습문제

1. 본인 도메인용 probe 30개를 직접 작성하라. 5개 카테고리 × 6 prompt. expected output 도 같이.
2. 본 책 10M 모델로 HellaSwag-tiny 30 문항을 돌려 accuracy 측정. 무작위 25% 대비 얼마나 위?
3. 도메인 probe 에서 `n=1` (greedy) vs `n=5` (pass@5) 의 통과율 차이는?
4. LLM judge (Haiku) 와 본인 평가의 상관계수 (50 샘플) 를 측정. r > 0.7 이면 judge 신뢰 가능.
5. **(생각해볼 것)** "정답이 1개" 인 probe vs "여러 답이 가능" 한 probe — 본인 도메인에 어느 쪽이 많은가? pass@k 가 의미 있는 도메인은?

In [ ]:
# 연습 1: 본인 도메인 probe 30개 작성


In [ ]:
# 연습 2: HellaSwag-tiny accuracy 측정


In [ ]:
# 연습 3: n=1 vs n=5 통과율 비교


In [ ]:
# 연습 4: LLM judge vs 사람 평가 상관계수


In [ ]:
# 연습 5: probe 유형 분석 (정답 1개 vs 여러 답)
